# ML-08 — Capstone Modeling Lane

**Lane:** Refresh / Content Opportunity Scoring  
**Goal:** learn a review-priority score and compare it honestly with the Week-4 hand-written baseline.

This notebook uses the real 30,000-row starter dataset. `trend_direction` is used only to create the evaluation proxy; `trend_direction` and `trend_pct` are never model features. Client/content IDs are used only for grouping and traceability.


## 1. Method choice and why

My question is **“which pages should an editor review first?”**, so I need a score that can rank pages. I start with **Logistic Regression** because its probability output is a simple ranking score and its coefficients are readable. I also test a **Random Forest** as a controlled nonlinear alternative.

The observed evaluation proxy is `trend_direction == "down"`. This is not a causal label: it describes measured direction in the starter snapshot and does not prove that refreshing a page will improve traffic.

I use five non-label inputs: `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, and `days_since_last_update`. This is intentionally small. Complexity only earns a place if it improves the same held-out ranking metric used for the baseline.


In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Starter dataset not found.")

df = pd.read_csv(DATA_PATH)
df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
]
forbidden = {"trend_direction", "trend_pct", "decline_proxy", "content_id", "client_id"}
assert set(features).isdisjoint(forbidden)

print(f"Rows: {len(df):,}")
print(f"Clients: {df['client_id'].nunique()}")
print(f"Observed decline-proxy rate: {df['decline_proxy'].mean():.1%}")
print("Five model features:", features)
print("scikit-learn:", sklearn.__version__)


Rows: 30,000
Clients: 32
Observed decline-proxy rate: 54.2%
Five model features: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'days_since_last_update']
scikit-learn: 1.8.0


## 2. Split design

I use a **grouped client split**: all pages from a client stay entirely in either train or test. This is more honest than a random row split because pages belonging to the same client can share measurement patterns and content practices.

The split is fixed with `random_state=42` for reproducibility. The Week-4 baseline and both learned models are evaluated on **exactly the same held-out test rows**.

The primary metric is **Precision@20**, matching the capacity-limited ranked-review decision. I also report Precision@50, AUROC, and Average Precision for context. The base rate is shown so a seemingly high score is not interpreted without a reference point.


In [2]:
X = df[features].copy()
y = df["decline_proxy"].astype(int)
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
assert train_clients.isdisjoint(test_clients)

print(f"Train: {len(train_idx):,} pages from {len(train_clients)} clients")
print(f"Test:  {len(test_idx):,} pages from {len(test_clients)} clients")
print(f"Train base rate: {y_train.mean():.1%}")
print(f"Test base rate:  {y_test.mean():.1%}")
print("Client overlap:", len(train_clients & test_clients))


Train: 22,885 pages from 24 clients
Test:  7,115 pages from 8 clients
Train base rate: 55.0%
Test base rate:  51.7%
Client overlap: 0


## 3. Train + compare vs my baseline

I rebuild the Week-4 baseline inside this notebook so the comparison uses the **same test rows and same metrics**.

**Week-4 baseline:** pages are eligible when they have at least 500 impressions and average position 1–20. The score combines an impression-impact percentile (60 points) with a low-CTR gap below 0.50% (40 points).

For the learned methods, Logistic Regression is the readable first model and Random Forest is the stronger nonlinear challenger. I do not choose the more complex model automatically; the comparison table decides whether its added complexity earns anything.


In [3]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

# ---- Baseline, fitted/calibrated only with test-row observable inputs; no label used ----
eligible_test = (
    (test_df["impressions_90d"] >= 500)
    & (test_df["avg_position"] > 0)
    & (test_df["avg_position"] <= 20)
)
impact = np.zeros(len(test_df))
impact[eligible_test.to_numpy()] = (
    np.log1p(test_df.loc[eligible_test, "impressions_90d"])
    .rank(pct=True, method="average")
    .to_numpy()
)
ctr_gap = np.zeros(len(test_df))
ctr_gap[eligible_test.to_numpy()] = np.clip(
    (0.50 - test_df.loc[eligible_test, "ctr"]) / 0.50, 0, 1
)
baseline_score = np.where(eligible_test, 60 * impact + 40 * ctr_gap, 0.0)

# ---- Logistic Regression ----
logit = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])
logit.fit(X_train, y_train)
logit_score = logit.predict_proba(X_test)[:, 1]

# ---- Random Forest ----
rf = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])
rf.fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

def row(name, score):
    return {
        "method": name,
        "Precision@20": precision_at_k(y_test, score, 20),
        "Precision@50": precision_at_k(y_test, score, 50),
        "AUROC": roc_auc_score(y_test, score),
        "Average Precision": average_precision_score(y_test, score),
    }

comparison = pd.DataFrame([
    {
        "method": "Base rate",
        "Precision@20": y_test.mean(),
        "Precision@50": y_test.mean(),
        "AUROC": 0.5,
        "Average Precision": y_test.mean(),
    },
    row("Week-4 baseline", baseline_score),
    row("Logistic Regression", logit_score),
    row("Random Forest", rf_score),
])

display(comparison.style.format({
    "Precision@20": "{:.1%}",
    "Precision@50": "{:.1%}",
    "AUROC": "{:.3f}",
    "Average Precision": "{:.3f}",
}))

best_name = comparison.iloc[1:].sort_values("Precision@20", ascending=False).iloc[0]["method"]
print("Best held-out Precision@20:", best_name)

# Save a small reproducibility receipt (JSON is allowed by the repo policy).
receipt = {
    "split": "GroupShuffleSplit by client_id, test_size=0.25, random_state=42",
    "train_rows": int(len(train_idx)),
    "test_rows": int(len(test_idx)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "features": features,
    "metrics": comparison.to_dict(orient="records"),
}
out_dir = DATA_PATH.resolve().parents[2] / "work/outputs"
try:
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "w05_model_metrics.json").write_text(json.dumps(receipt, indent=2))
    print("Wrote work/outputs/w05_model_metrics.json")
except PermissionError:
    print("Metrics receipt not written in this runtime; notebook results are still embedded.")


,method,Precision@20,Precision@50,AUROC,Average Precision
0,Base rate,51.7%,51.7%,0.500,0.517
1,Week-4 baseline,50.0%,58.0%,0.500,0.513
2,Logistic Regression,75.0%,62.0%,0.501,0.525
3,Random Forest,60.0%,72.0%,0.625,0.617


Best held-out Precision@20: Logistic Regression
Metrics receipt not written in this runtime; notebook results are still embedded.


## 4. Errors and interpretation

I inspect the **best learned model by Precision@20**, not just its headline score. Permutation importance is computed on the held-out test set, so it asks which feature most changes predictive performance when shuffled.

I also show three concrete high-confidence false positives: pages the model ranked strongly as declining even though the observed proxy was not down. These are useful because they reveal where the model's signals can be misleading.

Interpretation remains directional. Feature importance does not prove causality, and a predicted decline does not prove that a content refresh is the correct treatment.


In [4]:
# Choose the better learned model by the primary metric.
learned = {
    "Logistic Regression": (logit, logit_score),
    "Random Forest": (rf, rf_score),
}
learned_p20 = {name: precision_at_k(y_test, score, 20) for name, (_, score) in learned.items()}
chosen_name = max(learned_p20, key=learned_p20.get)
chosen_model, chosen_score = learned[chosen_name]

print("Chosen learned model for interpretation:", chosen_name)

perm = permutation_importance(
    chosen_model, X_test, y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)
importance = pd.DataFrame({
    "feature": features,
    "permutation_importance_mean": perm.importances_mean,
    "permutation_importance_std": perm.importances_std,
}).sort_values("permutation_importance_mean", ascending=False)

print("\nPermutation importance on held-out clients:")
display(importance)

errors = test_df[[
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "days_since_last_update", "decline_proxy"
]].copy()
errors["model_score"] = chosen_score

false_pos = (
    errors[errors["decline_proxy"] == 0]
    .sort_values("model_score", ascending=False)
    .head(3)
    .copy()
)
false_pos["why_hard"] = (
    "Signals resemble high-priority decline cases, but the observed proxy is not down; "
    "query intent, SERP effects, seasonality, or content context may explain the mismatch."
)
print("\nThree high-confidence false positives:")
display(false_pos)

# Error rates by visibility range.
errors["position_band"] = pd.cut(
    errors["avg_position"],
    bins=[-0.1, 0, 10, 20, 50, np.inf],
    labels=["no position data", "1-10", "11-20", "21-50", "50+"],
)
errors["predicted"] = (errors["model_score"] >= 0.5).astype(int)
errors["wrong"] = (errors["predicted"] != errors["decline_proxy"]).astype(int)

error_groups = (
    errors.groupby("position_band", observed=True)
    .agg(n=("content_id", "size"), error_rate=("wrong", "mean"))
    .reset_index()
)
print("\nWhere classification errors occur by position band:")
display(error_groups.style.format({"error_rate": "{:.1%}"}))

top3 = importance.head(3)["feature"].tolist()
print("\nTop three measured features:", ", ".join(top3))
print(
    "These features plausibly relate to visibility, click response, or content age, "
    "but their importance is predictive rather than causal."
)


Chosen learned model for interpretation: Logistic Regression


Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/spreadsheet_warmup.py", line 772, in warm_spreadsheet_runtime
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/rpc/connection.py", line 37, in get_or_create_client
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/rpc/daemon.py", line 124, in start_daemon
TimeoutError: Timed out waiting for artifact tool daemon socket. Set ARTIFACT_TOOL_RPC_DAEMON_STARTUP_TIMEOUT_S=<seconds> to increase the limit.
Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_

Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/spreadsheet_warmup.py", line 772, in warm_spreadsheet_runtime
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/rpc/connection.py", line 37, in get_or_create_client
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/rpc/daemon.py", line 124, in start_daemon
TimeoutError: Timed out waiting for artifact tool daemon socket. Set ARTIFACT_TOOL_RPC_DAEMON_STARTUP_TIMEOUT_S=<seconds> to increase the limit.
Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.L2TH2Y5coc/artifact_tool_v2-2.8.22/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_


Permutation importance on held-out clients:


,feature,permutation_importance_mean,permutation_importance_std
1,clicks_90d,0.018360,0.001160
3,avg_position,0.018004,0.002889
2,ctr,-0.000754,0.000923
0,impressions_90d,-0.001463,0.001597
4,days_since_last_update,-0.014750,0.006761



Three high-confidence false positives:


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,decline_proxy,model_score,why_hard
9476,content_30eb41dff556,client_d029fa3a95,84,0,0.0,6.2,183,0,0.691140,"Signals resemble high-priority decline cases, ..."
20924,content_277eeb6d46cc,client_d029fa3a95,17,0,0.0,7.9,183,0,0.689368,"Signals resemble high-priority decline cases, ..."
12801,content_2265b3e09778,client_d029fa3a95,12,0,0.0,8.4,183,0,0.688852,"Signals resemble high-priority decline cases, ..."



Where classification errors occur by position band:


,position_band,n,error_rate
0,no position data,64,3.1%
1,1-10,3668,53.5%
2,11-20,1732,49.4%
3,21-50,1372,55.0%
4,50+,279,55.9%



Top three measured features: clicks_90d, avg_position, ctr
These features plausibly relate to visibility, click response, or content age, but their importance is predictive rather than causal.


## Self-check

- [x] Method fits the lane: probability scores are used to rank pages.
- [x] Logistic Regression is the readable first model; Random Forest is tested only as a challenger.
- [x] Validation is grouped by `client_id`, with zero client overlap.
- [x] Random seed is fixed (`42`) for reproducibility.
- [x] Baseline and models are evaluated on the exact same held-out rows.
- [x] Primary metric is Precision@20; Precision@50, AUROC, Average Precision, and base rate are also reported.
- [x] Final comparison table includes base rate, Week-4 baseline, Logistic Regression, and Random Forest.
- [x] Five model features are used and no ID is a feature.
- [x] `trend_direction`, `trend_pct`, and the derived proxy are excluded from model inputs.
- [x] Permutation importance is reported on held-out clients.
- [x] Three concrete wrong cases and grouped error behavior are inspected.
- [x] Claims are observational/directional and do not imply that refreshing causes recovery.
- [x] Notebook runs top to bottom with no errors.
- [ ] Commit this executed notebook to `work/notebooks/w05_model.ipynb`, then submit the public repo URL.
